In [0]:
1==1

# Load Test History — Implementation Plan

## 1. Gap Analysis: What OTel Captures vs. What We Need

### What OTel already captures

| Signal | Data | Structured? | Queryable? |
| --- | --- | --- | --- |
| **Logs** | `[LoadTest/SSE] Starting: 500000 payloads across 5 type(s)` | No — free text | Regex/LIKE only |
| **Logs** | `[LoadTest/SSE] Complete: 1569600 records in 96530ms (16260 rec/s)` | No — free text | Regex/LIKE only |
| **Traces** | POST `/api/v1/testing/load-test/stream` span with `http.client_ip`, `duration` | Partially | Yes, but limited |
| **Metrics** | `http.server.duration` histogram for the POST endpoint | Yes | Aggregated only |

### What's missing from OTel (and can't be reconstructed)

| Field | Status | Why |
| --- | --- | --- |
| **User identity** (`x-forwarded-email`) | Not captured | OTel auto-instrumentation doesn't extract custom headers into span attributes |
| **Preset label** (Smoke/Small/Medium/Large/Massive) | Not captured | Client-side UI state, never sent to server |
| **Per-type counts configuration** (e.g., samples: 200, workouts: 100) | Not captured | In the POST request body, not logged structurally |
| **Batch size** | Not captured | Request body parameter |
| **Per-type results breakdown** (records per type, duration per type) | Not captured | Only in SSE progress events, not in logs |
| **Pool size at start/end** | Not captured | In-memory state only |
| **Auto-scale enabled?** | Partially | Log line exists, but not correlated to a run |
| **Run ID** | Not captured | No concept of a test run identifier |

**Verdict: OTel is insufficient.** The logs are unstructured text, the traces lack user identity and config, and metrics are aggregated counters — none provide the discrete, structured test-run records you need for a timeline table.

---

## 1b. Existing Bug: `user_id` on Synthetic Records

### Current state

The load test path **hardcodes** `user_id: 'synthetic-load-test'` on every generated bronze record:

| Layer | Code | Value |
| --- | --- | --- |
| **Client** (`LoadTestPage.tsx` line 267) | `userId: 'synthetic-load-test'` | Hardcoded in POST body |
| **Server** (`load-test-routes.ts` line 217) | `userId: body.userId ?? 'synthetic-load-test'` | Trusts client value |
| **Service** (`synthetic-data-service.ts`) | Passes `userId` to `zeroBusService.buildRecord()` | Flows through unchanged |
| **Bronze table** (`wearables_zerobus`) | `user_id` column | Every synthetic record = `'synthetic-load-test'` |

Meanwhile, the real HealthKit ingest path (`ingest-routes.ts`) correctly extracts the user via `extractUserFromToken(req)` which reads the trustworthy `x-forwarded-email` header injected by AppKit's reverse proxy.

### What this means

All \~6M+ synthetic records in the bronze table today have `user_id = 'synthetic-load-test'` instead of `matthew.giglia@databricks.com`. This makes it impossible to attribute load test runs to specific users at the data layer.

### Fix (included in this plan)

1. **Extract `x-forwarded-email` server-side** in `load-test-routes.ts` — don't trust the client-supplied `userId`
2. **Move `extractUserFromToken()` to a shared utility** (`server/utils/extract-user.ts`) so both `ingest-routes.ts` and `load-test-routes.ts` use the same logic
3. **Pass the real user identity** through to `syntheticDataService.generateAndIngestStreaming()` as the `userId` parameter
4. **Remove `userId` from the client POST body** in `LoadTestPage.tsx` — identity is a server concern, not a client one

This ensures every synthetic record's `user_id` column matches the workspace user who triggered the test — consistent with real HealthKit ingest and queryable for attribution.

---

## 2. Architecture Options Evaluated

| Option | Pros | Cons | Lakehouse? |
| --- | --- | --- | --- |
| **A. Custom OTel metrics** | Uses existing pipeline, no new infra | Metrics are aggregated time-series, not discrete run records; can't store per-run metadata (user, preset, per-type breakdown) | Partial (metrics table) |
| **B. Structured OTel logs** (custom log records) | Data lands in UC via existing exporter | Still text-based in the `body` VARIANT column; no schema enforcement; hard to query per-type breakdown | Yes (logs table) |
| **C. Lakebase only** | Fast CRUD, structured schema, Postgres JSON support, already wired, proven pattern (todo-routes.ts) | Data isolated from lakehouse; can't join with bronze table or SDP outputs | No |
| **D. ZeroBus to bronze table only** | Data in lakehouse, reuses existing pipeline | Slow reads for UI (no direct query from React); overkill for small dataset; schema-on-read makes querying harder | Yes |
| **E. Lakebase + ZeroBus dual-write** | Fast UI reads + lakehouse analytics; best of both worlds; clean separation of concerns | Two write paths to maintain | **Yes — both** |

---

## 3. Recommended Architecture: Lakebase + Lakehouse Sync

```
  LoadTestPage (React)
    │
    ▼ POST /api/v1/testing/load-test/stream
  load-test-routes.ts
    │  ← receives SSE "complete" result from syntheticDataService
    │
    └──► Lakebase (Postgres)                  ← single write target
         ├── app.load_test_runs                ← one row per test run
         └── app.load_test_type_results        ← one row per type per run
              │
              │  Lakehouse Sync (native CDC, automatic)
              │  wal2delta extension → WAL capture → Delta append
              ▼
         Unity Catalog (Delta tables, auto-created)
         ├── lb_load_test_runs_history          ← SCD Type 2 change log
         └── lb_load_test_type_results_history  ← SCD Type 2 change log
              │
              ▼
         Current-state views (DDL in infra bundle)
         ├── v_load_test_runs                   ← latest row per run_id
         └── v_load_test_type_results           ← latest row per (run_id, record_type)
```

### Why this replaces the dual-write approach

1. **Single write path** — the app writes to Lakebase only. No ZeroBus dual-write, no two code paths to maintain, no risk of the two stores drifting
2. **Lakehouse Sync** is a native Lakebase feature (Beta) that uses CDC (WAL logical decoding via the `wal2delta` Postgres extension) to continuously replicate every INSERT/UPDATE/DELETE to UC Delta tables — no external compute, pipelines, or jobs required
3. **SCD Type 2 history** — every change is appended with `_change_type`, `_timestamp`, `_lsn`, `_xid` system columns, giving full audit trail in the lakehouse
4. **Auto-created tables** — Lakehouse Sync creates `lb_<table_name>_history` Delta tables automatically in the configured UC catalog/schema. No target DDL needed for the history tables themselves
5. **Keeps `wearables_zerobus` pure** — load test metadata lives in its own Lakebase tables and its own UC Delta tables, completely separate from the real ingest bronze table
6. **Current-state views** in UC make querying easy — a simple view that deduplicates the SCD history by taking the latest non-delete row per primary key

### Lakehouse Sync — how it works

| Aspect | Detail |
| --- | --- |
| Mechanism | `wal2delta` Postgres extension captures WAL changes via logical decoding |
| Granularity | Schema-level — all current and future tables in the `app` schema sync automatically |
| Naming | `lb_<table_name>_history` in the destination UC catalog/schema |
| Change types | `insert`, `delete`, `update_preimage`, `update_postimage` |
| System columns | `_change_type` (TEXT), `_timestamp` (TIMESTAMP), `_lsn` (BIGINT), `_xid` (INTEGER) |
| Latency | Continuous, low-latency (seconds) |
| Compute | None — runs inside Lakebase, no Spark cluster or job |

### Prerequisites for Lakehouse Sync

| Requirement | Action | Owner |
| --- | --- | --- |
| **Preview enabled** | Workspace admin enables "Lakebase Change Data Feed" on the Previews page | Admin |
| **REPLICA IDENTITY FULL** | `ALTER TABLE app.load_test_runs REPLICA IDENTITY FULL;` (and same for type_results) — must be set **before** first INSERT | App startup (in `load-test-history-service.ts` table setup) |
| **UC permissions** | `USE CATALOG`, `USE SCHEMA`, `CREATE TABLE` on the destination catalog/schema | Infra bundle (existing SPN grants cover this) |
| **Database** | Tables must be in `databricks_postgres` database (Lakebase default) | Already the case |
| **Start sync** | Configure in Lakebase UI: select `app` schema → destination catalog/schema → Start sync | One-time manual step (or scripted via API) |

### Data type consideration: UUID → TEXT

Lakehouse Sync's supported type list does not explicitly include Postgres `UUID`. To avoid sync failures, the Lakebase schema uses `TEXT` for `run_id` instead of `UUID`, with `gen_random_uuid()::TEXT` as the default. This maps cleanly to `STRING` in Delta.

### Preset label

Can now be sent from the client to the server (add to the POST body), solving the missing metadata gap.

---

## 4. Schema Design

### Lakebase: `app.load_test_runs`

```sql
CREATE TABLE IF NOT EXISTS app.load_test_runs (
  run_id            TEXT PRIMARY KEY DEFAULT gen_random_uuid()::TEXT,  -- TEXT not UUID (Lakehouse Sync compat)
  user_id           TEXT NOT NULL,                  -- x-forwarded-email or 'anonymous'
  user_ip           TEXT,                           -- http.client_ip for audit
  started_at        TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  completed_at      TIMESTAMPTZ,
  duration_ms       INT,
  status            TEXT NOT NULL DEFAULT 'running', -- running | complete | error | aborted
  error_message     TEXT,

  -- Configuration
  preset_label      TEXT,                           -- Smoke | Small | Medium | Large | Massive | Custom
  batch_size        INT NOT NULL,
  total_payloads    INT NOT NULL,

  -- Results
  total_records     INT,
  records_per_sec   INT,

  -- Pool state
  pool_size_start   INT,
  pool_size_end     INT,
  auto_scale_enabled BOOLEAN DEFAULT false,
  auto_scale_min    INT,
  auto_scale_max    INT
);
```

### Lakebase: `app.load_test_type_results`

```sql
CREATE TABLE IF NOT EXISTS app.load_test_type_results (
  run_id            TEXT NOT NULL REFERENCES app.load_test_runs(run_id) ON DELETE CASCADE,
  record_type       TEXT NOT NULL,           -- samples | workouts | sleep | activity_summaries | deletes
  payload_count     INT NOT NULL,            -- configured count for this type
  record_count      INT,                     -- actual records ingested
  duration_ms       INT,
  records_per_sec   INT,
  PRIMARY KEY (run_id, record_type)
);
```

### UC Delta tables (auto-created by Lakehouse Sync)

Lakehouse Sync creates these automatically when sync is started — no DDL needed:

* `lb_load_test_runs_history` — every INSERT/UPDATE/DELETE on `app.load_test_runs`
* `lb_load_test_type_results_history` — every INSERT/UPDATE/DELETE on `app.load_test_type_results`

Each row includes the full Postgres row data plus system columns:

| System Column | Type | Description |
| --- | --- | --- |
| `_change_type` | STRING | `insert`, `delete`, `update_preimage`, `update_postimage` |
| `_timestamp` | TIMESTAMP | Postgres transaction commit time |
| `_lsn` | BIGINT | Postgres Log Sequence Number |
| `_xid` | INT | Postgres Transaction ID |

### UC current-state views (DDL in infra bundle `target-table-ddl` notebook)

The SCD Type 2 history tables contain every change event. For easy querying, we add views that deduplicate to the latest state:

```sql
-- Latest state of each load test run (excludes deleted rows)
CREATE OR REPLACE VIEW v_load_test_runs AS
SELECT * EXCEPT (_change_type, _timestamp, _lsn, _xid)
FROM (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY run_id ORDER BY _lsn DESC, _timestamp DESC
  ) AS _rn
  FROM lb_load_test_runs_history
  WHERE _change_type != 'update_preimage'
)
WHERE _rn = 1 AND _change_type != 'delete';

-- Latest state of each per-type result (excludes deleted rows)
CREATE OR REPLACE VIEW v_load_test_type_results AS
SELECT * EXCEPT (_change_type, _timestamp, _lsn, _xid)
FROM (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY run_id, record_type ORDER BY _lsn DESC, _timestamp DESC
  ) AS _rn
  FROM lb_load_test_type_results_history
  WHERE _change_type != 'update_preimage'
)
WHERE _rn = 1 AND _change_type != 'delete';
```

---

## 5. Server-Side Changes

| File | Change | Purpose |
| --- | --- | --- |
| `server/utils/extract-user.ts` | **New file** — shared user identity extraction | Extracts `x-forwarded-email` from AppKit proxy headers; used by both ingest-routes and load-test-routes |
| `server/services/load-test-history-service.ts` | **New file** — Lakebase CRUD for load test runs | Table setup, insert run, update run, insert type results, query history |
| `server/routes/testing/load-test-routes.ts` | Extract real user_id server-side; add history write after SSE complete; add GET `/api/v1/testing/history` endpoint | Write to Lakebase on test completion (Lakehouse Sync handles UC replication); serve history to frontend |
| `server/routes/zerobus/ingest-routes.ts` | Import `extractUser` from shared utility instead of local function | Dedup: remove local `extractUserFromToken()`, use shared import |
| `shared/synthetic-healthkit.ts` | No changes | Already exports `RECORD_TYPES` used by both sides |
| `server/services/zerobus-service.ts` | No changes | `buildRecord()` + `ingestRecords()` already available |

### New endpoints

| Method | Path | Purpose |
| --- | --- | --- |
| `GET` | `/api/v1/testing/history` | Return paginated history (joins `load_test_runs` + `load_test_type_results`) |
| `GET` | `/api/v1/testing/history/:runId` | Return single run detail with per-type breakdown |
| `DELETE` | `/api/v1/testing/history/:runId` | Delete a run (cascades to type results) |

### Write flow (inside `load-test-routes.ts`)

```
1. Client POSTs to /load-test/stream with { counts, batchSize, presetLabel }
   (NO userId — identity is extracted server-side from x-forwarded-email)
2. Before SSE starts:
   a. Extract user_id via shared extractUser(req) → x-forwarded-email || 'anonymous'
   b. Pass user_id as userId to syntheticDataService (stamps every bronze record)
   c. Capture pool status (pool_size_start, auto_scale state)
   d. INSERT into app.load_test_runs (status='running', user_id=...) → get run_id
   e. INSERT into app.load_test_type_results (one per configured type, record_count=0)
3. On each SSE progress event:
   a. UPDATE type results with latest cumulative counts (optional — could batch)
4. On SSE complete:
   a. UPDATE app.load_test_runs SET status='complete', total_records=..., etc.
   b. UPDATE app.load_test_type_results with final per-type metrics
   c. (No dual-write needed — Lakehouse Sync replicates the Lakebase update to UC automatically)
5. On error/abort:
   a. UPDATE app.load_test_runs SET status='error', error_message=...
```

---

## 6. Client-Side Changes

| File | Change | Purpose |
| --- | --- | --- |
| `LoadTestPage.tsx` | Add `presetLabel` to POST body; **remove `userId` from POST body**; add History section below results | Send preset name to server; identity is server-side; render history UI |
| `LoadTestPage.tsx` (or new `LoadTestHistory.tsx`) | **New component** — history timeline table + charts | Fetch from `/api/v1/testing/history`, render table + visualizations |

### History UI Design

**Timeline Table** (sortable, most recent first):

| Date/Time | User | Preset | Records | Throughput | Duration | Pool | Status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Apr 22 17:17 | matthew.giglia@ | Massive | 1.57M | 16.3K/s | 1m 37s | 2→8 (auto) | Complete |
| Apr 22 15:50 | matthew.giglia@ | Large | 157K | 8.8K/s | 18s | 2→4 (auto) | Complete |

**Visualizations** (expandable or always visible):

1. **Throughput over time** — bar chart: each run's records_per_sec, colored by preset tier
2. **Records by type** — stacked bar: per-type breakdown for each run
3. **Pool scaling correlation** — scatter: pool_size_end vs. throughput, showing auto-scale effectiveness
4. **Duration by scale** — bar: duration_ms grouped by preset label

---

## 7. Implementation Order

| Step | Task | Files / Location |
| --- | --- | --- |
| 0 | **Extract shared `extractUser()` utility**; refactor `ingest-routes.ts` to use it; wire into `load-test-routes.ts` so real user identity flows to every synthetic bronze record | `server/utils/extract-user.ts`, `ingest-routes.ts`, `load-test-routes.ts`, `LoadTestPage.tsx` |
| 1 | Create `load-test-history-service.ts` with Lakebase schema + CRUD. Table setup includes `ALTER TABLE ... REPLICA IDENTITY FULL` for Lakehouse Sync compatibility | `server/services/` |
| 2 | Add `presetLabel` to `LoadTestRequest` interface and POST body; remove `userId` from client | `load-test-routes.ts`, `LoadTestPage.tsx` |
| 3 | Wire history writes into the SSE streaming flow (Lakebase only — Lakehouse Sync handles UC) | `load-test-routes.ts` |
| 4 | Add `GET /api/v1/testing/history` endpoint | `load-test-routes.ts` |
| 5 | **Add current-state view DDL** to the infra bundle's `target-table-ddl` notebook (runs after Lakehouse Sync creates the history tables) | `dbxW_zerobus_infra/src/uc_setup/target-table-ddl` (notebook ID `2003193682514586`) |
| 6 | **Enable Lakehouse Sync** in the Lakebase UI: source schema `app` → destination `${var.catalog}.${var.schema}` → Start sync | Lakebase Postgres UI (one-time manual step per target) |
| 7 | Build `LoadTestHistory` React component (timeline table + charts) | `client/src/pages/testing/` |
| 8 | Integrate history component into `LoadTestPage.tsx` | `LoadTestPage.tsx` |
| 9 | Test end-to-end: run a test → verify Lakebase rows → verify `lb_*_history` Delta tables → verify views → verify history UI |

---

## 8. Data Availability Summary

| Store | Query Method | Latency | Use Case |
| --- | --- | --- | --- |
| **Lakebase** (Postgres) | `appkit.lakebase.query()` from Express | <5ms | React UI history page — timeline table, charts |
| **UC History Tables** (`lb_*_history`) | Spark SQL, AI/BI | Seconds (continuous CDC sync) | Full SCD Type 2 audit trail — every insert, update, delete |
| **UC Current-State Views** (`v_load_test_*`) | Spark SQL, AI/BI, dashboards | Seconds | Deduplicated latest state — join with ingest data, build dashboards |
| **UC Gold Table** (future SDP) | Spark SQL, AI/BI | Seconds | Aggregated benchmarks, trend analysis (reads from views or history) |
| **OTel Logs** (existing) | Spark SQL | Seconds | Correlation / audit trail (unstructured, supplementary) |